In [1]:
%load_ext jupyter_black

In [2]:
from linear_regression import *
from common import results_folder

In [3]:
lr = LinearRegression()

In [4]:
coefs_rm = lr.generate_rm_results()

Loading RM 2021, ejected 0 for gpa availability, 30 for having 75% of ects available, resulting in 322
Loading RM 2022, ejected 0 for gpa availability, 3 for having 75% of ects available, resulting in 295
Loading RM 2023, ejected 0 for gpa availability, 7 for having 75% of ects available, resulting in 285


In [5]:
coefs_rm_dm = lr.generate_rm_dm_results()

Loading RM 2021_DA 2022, ejected 0 for gpa availability, 4 for having 75% of ects available, resulting in 282
Loading RM 2022_DA 2023, ejected 0 for gpa availability, 1 for having 75% of ects available, resulting in 255
Loading RM 2023_DA 2024, ejected 0 for gpa availability, 6 for having 75% of ects available, resulting in 260


In [6]:
lr.report_classify_per_cohort_rm()

RM 2021
                               OLS Regression Results                              
Dep. Variable:     gpa_last_attempt_year_1   R-squared:                       0.655
Model:                                 OLS   Adj. R-squared:                  0.653
Method:                      Least Squares   F-statistic:                     302.7
Date:                     Fri, 24 Apr 2026   Prob (F-statistic):           1.99e-74
Time:                             18:32:06   Log-Likelihood:                -240.78
No. Observations:                      322   AIC:                             487.6
Df Residuals:                          319   BIC:                             498.9
Df Model:                                2                                         
Covariance Type:                 nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

In [7]:
lr.report_classify_per_cohort_rm_dm()

RM 2021_DA 2022
                              OLS Regression Results                              
Dep. Variable:     gpa_last_attempt_total   R-squared:                       0.744
Model:                                OLS   Adj. R-squared:                  0.742
Method:                     Least Squares   F-statistic:                     404.9
Date:                    Fri, 24 Apr 2026   Prob (F-statistic):           3.18e-83
Time:                            18:32:06   Log-Likelihood:                -136.42
No. Observations:                     282   AIC:                             278.8
Df Residuals:                         279   BIC:                             289.8
Df Model:                               2                                         
Covariance Type:                nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------

In [8]:
lr.report_classify_rm_total()

                               OLS Regression Results                              
Dep. Variable:     gpa_last_attempt_year_1   R-squared:                       0.664
Model:                                 OLS   Adj. R-squared:                  0.663
Method:                      Least Squares   F-statistic:                     888.9
Date:                     Fri, 24 Apr 2026   Prob (F-statistic):          9.93e-214
Time:                             18:32:06   Log-Likelihood:                -717.93
No. Observations:                      902   AIC:                             1442.
Df Residuals:                          899   BIC:                             1456.
Df Model:                                2                                         
Covariance Type:                 nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

In [9]:
lr.report_classify_rm_dm_total()

                              OLS Regression Results                              
Dep. Variable:     gpa_last_attempt_total   R-squared:                       0.745
Model:                                OLS   Adj. R-squared:                  0.744
Method:                     Least Squares   F-statistic:                     1158.
Date:                    Fri, 24 Apr 2026   Prob (F-statistic):          4.27e-236
Time:                            18:32:06   Log-Likelihood:                -416.64
No. Observations:                     797   AIC:                             839.3
Df Residuals:                         794   BIC:                             853.3
Df Model:                               2                                         
Covariance Type:                nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------

In [10]:
from IPython.display import display


index_rename = {
    "RM 2021": "2020-2021ᵃ (n=322)",
    "RM 2022": "2021-2022 (n=295)",
    "RM 2023": "2022-2023 (n=285)",
    "RM Total": "Total (n=902)",
    "RM 2021_DA 2022": "2020-2021 (n=282)",
    "RM 2022_DA 2023": "2021-2022 (n=255)",
    "RM 2023_DA 2024": "2022-2023 (n=260)",
    "RM DA Total": "Total (n=797)",
}


# B
# SE
# B
# t
# P
# 95% CI
def transform_coefs(coefs):
    cols = list(set([x for x, _, _ in coefs.columns]))
    rename = {
        x: "Last sitting grades" if "last" in x else "First sitting grades"
        for x in cols
    }
    print(rename)
    coefs = coefs.copy()
    coefs = coefs.rename(columns=rename, level=0)
    coefs = coefs.rename(
        columns={"const": "Intercept", "VSAQ": "VSAQ z-score", "MCQ": "MCQ z-score"},
        level=1,
    )

    def format_odd(row):
        # print(row)
        odds = row.xs("coef", level=2).iloc[0]
        se = row.xs("se", level=2).iloc[0]
        lower = row.xs("5%", level=2).iloc[0]
        upper = row.xs("95%", level=2).iloc[0]
        t = row.xs("t", level=2).iloc[0]
        p = row.xs("p-values", level=2).iloc[0]

        return pd.Series(
            [
                f"{odds:.2f}",
                # f"{se:.2f}",
                f"{t:.2f}",
                # f"{p:.2f}",
                f"[{lower:.2f}–{upper:.2f}]",
            ],
            index=["ß", "t", "95% CI"],
        )

    def format_odds(row):
        return row.apply(format_odd)
        return f""

    return (
        coefs.T.groupby(level=[0, 1])
        .apply(format_odds)
        .T[["Last sitting grades", "First sitting grades"]]
        .rename(index=index_rename)
    )
    return coefs


pd.concat(
    [transform_coefs(coefs_rm), transform_coefs(coefs_rm_dm)],
    keys=["RM Participants", "RM & DA participants"],
).loc[
    (slice(None), index_rename.values()),
    (
        ["First sitting grades", "Last sitting grades"],
        ["VSAQ z-score", "MCQ z-score"],
        slice(None),
    ),
].to_excel(
    results_folder / "linear.xlsx"
)

{'gpa_last_attempt_year_1': 'Last sitting grades', 'gpa_first_attempt_year_1': 'First sitting grades'}
{'gpa_last_attempt_total': 'Last sitting grades', 'gpa_first_attempt_total': 'First sitting grades'}
